# Estatística

  Para gerar a base de estatística, primeiramente foram importadas funções e constantes dos módulos constantes.py, aritmetica.py e funcoes.py e em alguns agoritmos foram importadas outras funções para uso momentâneo, desenvolvidos anteriormente.

  Como base, foram feitos algoritmos para medidas de tendência central, sendo para média, média ponderada, média geométrica, média harmônica, mediana e moda. em seguida foram implementadas medidas de dispersão por amplitude, variança populacional, variança amostral, desvio padrão populacional, desvio padrão amostral, desvio médio absoluto, coeficiente de variação, percentil, quartil, amplitude interquastil, detecção de outliers e um resumo com as medidas de mínimo, q1, mediana, q3 e máximo para análises comparativas. Também foram adicionados algoritmos para as medidas de forma assimetria e curtose.

  Para analisar a relação entre duas variáveis, foram implementados códigos que calculam a covariancia populacional, covariancia amostral, correlação de Pearson, correlação de Spearman, postos, regressão linear simples. predição de regrassão linear, regressão polinomial e avaliação da regressão polinomial.

  Foi definida uma classe chamada "TabelaDeFrequencias" para definir as tabelas de frequências como objetos dentro do Python. Essa classe define as partes que compõem uma tabela de frequência. Também foram definidas as maneiras com que é interpretada internamente, como ela aparecerá ao ser printada, como aparecerá internamente no código, dentre outras propriedades. Também foram definidas as propriedades dentro das tabelas de frequência, com algoritmos internos a ela que calculam frequências relativas, frequências acumuladas e média estimada.

  Para trabalhar com esses novos objetos, foram desenvolvidos algoritmos para testes de hipóteses. Sendo eles o teste z de uma amostra, com desvio populacional conhecido; teste t de uma amostra, com desvio populacional desconhecido; teste t de duas amostras, comparação das médias de duas amostras; e a probabilidade acumulada da normal padrão. Por fim foram definidos os intervalos de confiança da média, valor crítico normal aproximado, normalização de dados de z-score e min-max e a série temporal de média móvel.

  Para finalizar, adicionou-se a linha de código "%%writefile estatistica.py" para converter o código em um módulo exportável para ser usado em outros módulos e no uso final. Pelo mesmo motivo, o código teve que ser colocado inteiramente em uma única célula de código, pois o comando usado para isso exporta códigos de apenas uma célula para isso.

In [1]:
%%writefile estatistica.py

from constantes import tolerancia_de_erro, maximo_de_interacoes, pi
from aritmetica import valor_absoluto
from funcoes import sqrt, ln, exponencial, gamma

#Medidas de tendência central
#Média
def media(dados: list) -> float:
    if not dados:
        raise ValueError("Lista de dados não pode ser vazia.")
    return sum(dados) / len(dados)


#Média ponderada
def media_ponderada(dados: list, pesos: list) -> float:
    if len(dados) != len(pesos):
        raise ValueError("Dados e pesos devem ter o mesmo tamanho.")
    soma_pesos = sum(pesos)
    if soma_pesos == 0:
        raise ZeroDivisionError("Soma dos pesos não pode ser zero.")
    return sum(d * p for d, p in zip(dados, pesos)) / soma_pesos


#Média geométrica
def media_geometrica(dados: list) -> float:
    if not dados:
        raise ValueError("Lista de dados não pode ser vazia.")
    for d in dados:
        if d <= 0:
            raise ValueError("Média geométrica requer todos os valores positivos.")
    soma_log = sum(ln(d) for d in dados)
    return exponencial(soma_log / len(dados))


#Média harmônica
def media_harmonica(dados: list) -> float:
    if not dados:
        raise ValueError("Lista de dados não pode ser vazia.")
    for d in dados:
        if d == 0:
            raise ZeroDivisionError("Média harmônica não definida com valor zero na lista.")
    return len(dados) / sum(1.0 / d for d in dados)


#Mediana
def mediana(dados: list) -> float:
    if not dados:
        raise ValueError("Lista de dados não pode ser vazia.")
    ordenados = sorted(dados)
    n = len(ordenados)
    meio = n // 2
    if n % 2 == 0:
        return (ordenados[meio - 1] + ordenados[meio]) / 2.0
    return float(ordenados[meio])


#Moda
def moda(dados: list) -> list:
    if not dados:
        raise ValueError("Lista de dados não pode ser vazia.")
    contagem = {}
    for d in dados:
        contagem[d] = contagem.get(d, 0) + 1
    frequencia_maxima = max(contagem.values())
    return sorted([valor for valor, freq in contagem.items() if freq == frequencia_maxima])


#Medidas de dispersão
#Amplitude
def amplitude(dados: list) -> float:
    if not dados:
        raise ValueError("Lista de dados não pode ser vazia.")
    return max(dados) - min(dados)


#Variância populacional
def variancia_populacional(dados: list) -> float:
    if not dados:
        raise ValueError("Lista de dados não pode ser vazia.")
    m = media(dados)
    return sum((x - m) ** 2 for x in dados) / len(dados)


#Variância amostral
def variancia_amostral(dados: list) -> float:
    n = len(dados)
    if n < 2:
        raise ValueError("Variância amostral requer ao menos 2 valores.")
    m = media(dados)
    return sum((x - m) ** 2 for x in dados) / (n - 1)


#Desvio padrão populacional
def desvio_padrao_populacional(dados: list) -> float:
    return sqrt(variancia_populacional(dados))


#Desvio padrão amostral
def desvio_padrao_amostral(dados: list) -> float:
    return sqrt(variancia_amostral(dados))


#Desvio médio absoluto
def desvio_medio_absoluto(dados: list) -> float:
    if not dados:
        raise ValueError("Lista de dados não pode ser vazia.")
    m = media(dados)
    return sum(valor_absoluto(x - m) for x in dados) / len(dados)


#Coeficiente de variação
def coeficiente_de_variacao(dados: list) -> float:
    m = media(dados)
    if valor_absoluto(m) < tolerancia_de_erro:
        raise ZeroDivisionError("Coeficiente de variação indefinido quando a média é zero.")
    return desvio_padrao_amostral(dados) / m


#Percentil
def percentil(dados: list, p: float) -> float:
    if not (0 <= p <= 100):
        raise ValueError("Percentil p deve estar entre 0 e 100.")
    if not dados:
        raise ValueError("Lista de dados não pode ser vazia.")

    ordenados = sorted(dados)
    n = len(ordenados)
    if n == 1:
        return float(ordenados[0])

    posicao = (p / 100.0) * (n - 1)
    indice_inferior = int(posicao)
    indice_superior = min(indice_inferior + 1, n - 1)
    fracao = posicao - indice_inferior

    return ordenados[indice_inferior] + fracao * (ordenados[indice_superior] - ordenados[indice_inferior])


#Quartil
def quartil(dados: list, q: int) -> float:
    if q not in (1, 2, 3):
        raise ValueError("q deve ser 1, 2 ou 3 (primeiro, segundo ou terceiro quartil).")
    return percentil(dados, q * 25.0)


#Amplitude interquartil
def amplitude_interquartil(dados: list) -> float:
    return quartil(dados, 3) - quartil(dados, 1)


#Detecção de outliers
def deteccao_de_outliers(dados: list) -> list:
    q1 = quartil(dados, 1)
    q3 = quartil(dados, 3)
    iqr = q3 - q1
    limite_inferior = q1 - 1.5 * iqr
    limite_superior = q3 + 1.5 * iqr
    return [x for x in dados if x < limite_inferior or x > limite_superior]


#Resumo dos cinco números
def resumo_dos_cinco_numeros(dados: list) -> dict:
    return {
        "minimo": min(dados),
        "q1": quartil(dados, 1),
        "mediana": mediana(dados),
        "q3": quartil(dados, 3),
        "maximo": max(dados),
    }


#Medidas de forma
#Assimetria
def assimetria(dados: list) -> float:
    n = len(dados)
    if n < 3:
        raise ValueError("Assimetria requer ao menos 3 valores.")
    m = media(dados)
    s = desvio_padrao_amostral(dados)
    if s < tolerancia_de_erro:
        raise ZeroDivisionError("Assimetria indefinida quando o desvio padrão é zero.")
    soma_cubos = sum((x - m) ** 3 for x in dados)
    return (n / ((n - 1) * (n - 2))) * (soma_cubos / (s ** 3))


#Curtose
def curtose(dados: list) -> float:
    n = len(dados)
    if n < 4:
        raise ValueError("Curtose requer ao menos 4 valores.")
    m = media(dados)
    s = desvio_padrao_amostral(dados)
    if s < tolerancia_de_erro:
        raise ZeroDivisionError("Curtose indefinida quando o desvio padrão é zero.")
    soma_quartas = sum((x - m) ** 4 for x in dados)
    termo1 = (n * (n + 1)) / ((n - 1) * (n - 2) * (n - 3))
    termo2 = soma_quartas / (s ** 4)
    termo3 = (3 * (n - 1) ** 2) / ((n - 2) * (n - 3))
    return termo1 * termo2 - termo3


#Relação entre duas variáveis
#Covariância populacional
def covariancia_populacional(dados_x: list, dados_y: list) -> float:
    if len(dados_x) != len(dados_y):
        raise ValueError("dados_x e dados_y devem ter o mesmo tamanho.")
    mx = media(dados_x)
    my = media(dados_y)
    n = len(dados_x)
    return sum((x - mx) * (y - my) for x, y in zip(dados_x, dados_y)) / n


#Covariância amostral
def covariancia_amostral(dados_x: list, dados_y: list) -> float:
    n = len(dados_x)
    if n < 2:
        raise ValueError("Covariância amostral requer ao menos 2 pares de valores.")
    mx = media(dados_x)
    my = media(dados_y)
    return sum((x - mx) * (y - my) for x, y in zip(dados_x, dados_y)) / (n - 1)


#Correlação de Pearson
def correlacao_de_pearson(dados_x: list, dados_y: list) -> float:
    if len(dados_x) != len(dados_y):
        raise ValueError("Dados de x e de y devem ter o mesmo tamanho.")
    sx = desvio_padrao_amostral(dados_x)
    sy = desvio_padrao_amostral(dados_y)
    if sx < tolerancia_de_erro or sy < tolerancia_de_erro:
        raise ZeroDivisionError("Correlação indefinida quando desvio padrão de x ou y é zero.")
    cov = covariancia_amostral(dados_x, dados_y)
    return cov / (sx * sy)


#Correlação de Spearman
def correlacao_de_spearman(dados_x: list, dados_y: list) -> float:
    if len(dados_x) != len(dados_y):
        raise ValueError("Dados de x e de y devem ter o mesmo tamanho.")
    postos_x = calcular_postos(dados_x)
    postos_y = calcular_postos(dados_y)
    return correlacao_de_pearson(postos_x, postos_y)


#Definição dos postos
def calcular_postos(dados: list) -> list:
    n = len(dados)
    indices_ordenados = sorted(range(n), key=lambda i: dados[i])
    postos = [0.0] * n

    i = 0
    while i < n:
        j = i
        while j + 1 < n and dados[indices_ordenados[j + 1]] == dados[indices_ordenados[i]]:
            j += 1
        posto_medio = (i + j) / 2.0 + 1.0
        for k in range(i, j + 1):
            postos[indices_ordenados[k]] = posto_medio
        i = j + 1

    return postos


#Regressão linear simples
def regressao_linear_simples(dados_x: list, dados_y: list) -> dict:
    if len(dados_x) != len(dados_y):
        raise ValueError("Dados de x e y devem ter o mesmo tamanho.")

    n = len(dados_x)
    media_x = media(dados_x)
    media_y = media(dados_y)

    soma_xy = sum((x - media_x) * (y - media_y) for x, y in zip(dados_x, dados_y))
    soma_xx = sum((x - media_x) ** 2 for x in dados_x)

    if soma_xx < tolerancia_de_erro:
        raise ZeroDivisionError("Regressão linear indefinida, pois a variância de x é zero.")

    inclinacao = soma_xy / soma_xx
    intercepto = media_y - inclinacao * media_x

    previsoes = [intercepto + inclinacao * x for x in dados_x]
    residuos = [y - p for y, p in zip(dados_y, previsoes)]

    soma_quadrados_residuos = sum(r ** 2 for r in residuos)
    soma_quadrados_totais = sum((y - media_y) ** 2 for y in dados_y)

    r_quadrado = 1.0 - (soma_quadrados_residuos / soma_quadrados_totais) if soma_quadrados_totais > tolerancia_de_erro else 1.0

    return {
        "inclinacao": inclinacao,
        "intercepto": intercepto,
        "r_quadrado": r_quadrado,
        "residuos": residuos,
        "equacao": f"y = {inclinacao:.6f}x + {intercepto:.6f}",
    }

#Predição de regressão linear
def predizer_regressao(modelo: dict, x: float) -> float:
    return modelo["intercepto"] + modelo["inclinacao"] * x


#Regressão polinomial
def regressao_polinomial(dados_x: list, dados_y: list, grau: int) -> list:
    n = len(dados_x)
    if n != len(dados_y):
        raise ValueError("dados_x e dados_y devem ter o mesmo tamanho.")
    if grau >= n:
        raise ValueError("O grau deve ser menor que o número de pontos.")

    A = [[sum(x ** (i + j) for x in dados_x) for j in range(grau + 1)] for i in range(grau + 1)]
    b = [sum(y * (x ** i) for x, y in zip(dados_x, dados_y)) for i in range(grau + 1)]

    return _resolver_sistema_gauss(A, b)


def _resolver_sistema_gauss(A: list, b: list) -> list:
    n = len(A)
    M = [A[i][:] + [b[i]] for i in range(n)]

    for col in range(n):
        linha_pivo = col
        maior_valor = valor_absoluto(M[col][col])
        for r in range(col + 1, n):
            if valor_absoluto(M[r][col]) > maior_valor:
                maior_valor = valor_absoluto(M[r][col])
                linha_pivo = r
        M[col], M[linha_pivo] = M[linha_pivo], M[col]

        pivo = M[col][col]
        if valor_absoluto(pivo) < tolerancia_de_erro:
            raise ValueError("Sistema singular na regressão polinomial.")

        for r in range(col + 1, n):
            fator = M[r][col] / pivo
            for c in range(col, n + 1):
                M[r][c] -= fator * M[col][c]

    x = [0.0] * n
    for i in range(n - 1, -1, -1):
        s = M[i][n]
        for j in range(i + 1, n):
            s -= M[i][j] * x[j]
        x[i] = s / M[i][i]

    return x


#Avaliação da regressão polinomial
def avaliar_regressao_polinomial(coeficientes: list, x: float) -> float:
    resultado = 0.0
    for i, c in enumerate(coeficientes):
        resultado += c * (x ** i)
    return resultado


#Definição de classe para uma tabela de frequências
class TabelaDeFrequencias:

    def __init__(proprio, dados: list, numero_de_classes: int = None):
        proprio.dados = sorted(dados)
        n = len(dados)

        if numero_de_classes is None:
            numero_de_classes = max(1, int(sqrt(n) + 0.5))

        minimo = min(dados)
        maximo = max(dados)
        amplitude_total = maximo - minimo
        if amplitude_total < tolerancia_de_erro:
            amplitude_total = 1.0

        largura_classe = amplitude_total / numero_de_classes

        proprio.limites = [minimo + i * largura_classe for i in range(numero_de_classes + 1)]
        proprio.frequencias = [0] * numero_de_classes

        for valor in dados:
            indice = int((valor - minimo) / largura_classe)
            if indice >= numero_de_classes:
                indice = numero_de_classes - 1
            proprio.frequencias[indice] += 1

        proprio.pontos_medios = [
            (proprio.limites[i] + proprio.limites[i + 1]) / 2.0
            for i in range(numero_de_classes)
        ]

    def frequencias_relativas(proprio) -> list:
        total = sum(proprio.frequencias)
        return [f / total for f in proprio.frequencias]

    def frequencias_acumuladas(proprio) -> list:
        acumulado = []
        soma = 0
        for f in proprio.frequencias:
            soma += f
            acumulado.append(soma)
        return acumulado

    def media_estimada(proprio) -> float:
        total = sum(proprio.frequencias)
        return sum(pm * f for pm, f in zip(proprio.pontos_medios, proprio.frequencias)) / total

    def __str__(proprio) -> str:
        linhas = []
        for i in range(len(proprio.frequencias)):
            linhas.append(
                f"[{proprio.limites[i]:.4f}, {proprio.limites[i+1]:.4f}) : {proprio.frequencias[i]}"
            )
        return "\n".join(linhas)


#Testes de hipóteses
#Com desvio populacional conhecido
def teste_z_uma_amostra(dados: list, media_hipotese: float, desvio_populacional: float) -> dict:
    n = len(dados)
    m = media(dados)
    erro_padrao = desvio_populacional / sqrt(n)
    z = (m - media_hipotese) / erro_padrao
    p_valor = 2.0 * (1.0 - cdf_normal_padrao(valor_absoluto(z)))
    return {"estatistica_z": z, "p_valor": p_valor, "media_amostral": m}


#Com desvio populacional desconhecido
def teste_t_uma_amostra(dados: list, media_hipotese: float) -> dict:
    n = len(dados)
    if n < 2:
        raise ValueError("Teste t requer ao menos 2 valores.")
    m = media(dados)
    s = desvio_padrao_amostral(dados)
    erro_padrao = s / sqrt(n)
    if erro_padrao < tolerancia_de_erro:
        raise ZeroDivisionError("Erro padrão ≈ 0: teste t indefinido.")
    t = (m - media_hipotese) / erro_padrao
    graus_liberdade = n - 1
    return {"estatistica_t": t, "graus_liberdade": graus_liberdade, "media_amostral": m}


#Comparação das médias de duas amostras
def teste_t_duas_amostras(dados_x: list, dados_y: list) -> dict:
    n_x, n_y = len(dados_x), len(dados_y)
    if n_x < 2 or n_y < 2:
        raise ValueError("Teste t de duas amostras requer ao menos 2 valores em cada grupo.")

    mx, my = media(dados_x), media(dados_y)
    sx2, sy2 = variancia_amostral(dados_x), variancia_amostral(dados_y)

    variancia_combinada = (((n_x - 1) * sx2 + (n_y - 1) * sy2)) / (n_x + n_y - 2)
    erro_padrao = sqrt(variancia_combinada * (1.0 / n_x + 1.0 / n_y))

    if erro_padrao < tolerancia_de_erro:
        raise ZeroDivisionError("Erro padrão ≈ 0: teste t indefinido.")

    t = (mx - my) / erro_padrao
    graus_liberdade = n_x + n_y - 2

    return {"estatistica_t": t, "graus_liberdade": graus_liberdade, "media_x": mx, "media_y": my}


#Probabilidade acumulada da normal padrão
def cdf_normal_padrao(z: float) -> float:
    if z == 0.0:
        return 0.5
    x = z / sqrt(2.0)
    if valor_absoluto(x) > 6.0:
        erro = 1.0 if x > 0 else -1.0
    else:
        resultado = 0.0
        termo = x
        x2 = x * x
        for k in range(maximo_de_interacoes):
            resultado += termo / (2 * k + 1)
            proximo_termo = -termo * x2 / (k + 1)
            if valor_absoluto(proximo_termo) < tolerancia_de_erro * 1e-10:
                break
            termo = proximo_termo
        erro = (2.0 / sqrt(pi)) * resultado
    return 0.5 * (1.0 + erro)


#Intervalos de confiança
#Intervalo de confiança da média
def intervalo_de_confianca_media(dados: list, nivel_de_confianca: float = 0.95) -> tuple:
    n = len(dados)
    m = media(dados)
    s = desvio_padrao_amostral(dados)
    erro_padrao = s / sqrt(n)

    valor_critico = valor_critico_normal_aproximado(nivel_de_confianca)
    margem = valor_critico * erro_padrao

    return (m - margem, m + margem)


#Valor crítico normal aproximado
def valor_critico_normal_aproximado(nivel_de_confianca: float) -> float:
    alfa = 1.0 - nivel_de_confianca
    p_objetivo = 1.0 - alfa / 2.0

    z_baixo, z_alto = 0.0, 6.0
    for _ in range(100):
        z_meio = (z_baixo + z_alto) / 2.0
        if cdf_normal_padrao(z_meio) < p_objetivo:
            z_baixo = z_meio
        else:
            z_alto = z_meio
    return (z_baixo + z_alto) / 2.0


#Normalização de dados
def normalizar_z_score(dados: list) -> list:
    m = media(dados)
    s = desvio_padrao_amostral(dados)
    if s < tolerancia_de_erro:
        raise ZeroDivisionError("Normalização z-score indefinida quando desvio padrão é zero.")
    return [(x - m) / s for x in dados]


def normalizar_min_max(dados: list) -> list:
    minimo = min(dados)
    maximo = max(dados)
    amplitude_total = maximo - minimo
    if amplitude_total < tolerancia_de_erro:
        raise ZeroDivisionError("Normalização min-max indefinida quando todos os valores são iguais.")
    return [(x - minimo) / amplitude_total for x in dados]


#Série temporal - média móvel
def media_movel(dados: list, janela: int) -> list:
    if janela <= 0 or janela > len(dados):
        raise ValueError("Tamanho da janela inválido.")
    resultado = []
    for i in range(len(dados) - janela + 1):
        resultado.append(media(dados[i:i + janela]))
    return resultado

Overwriting estatistica.py
